<div align="center">

# 🚕 Yandex Go Analytics

## 3 · Streaming Ingestion & Delta Features

*Mock a live driver stream, evolve its schema on the fly, then travel back in time.*

![layer](https://img.shields.io/badge/layer-bronze-CD7F32)
![engine](https://img.shields.io/badge/engine-Structured%20Streaming-E25A1C)
![feature](https://img.shields.io/badge/feature-Time%20Travel-00ADD4)
![feature](https://img.shields.io/badge/feature-Schema%20Evolution-7B1FA2)

</div>

| ◀ Previous | 🏠 Home | Next ▶ |
|:--|:--:|--:|
| [2 · Batch Ingestion](./2.%20Batch%20Ingestion.ipynb) | [Project Overview](./0.%20Project%20Overview.ipynb) | [4 · Data Cleaning](./4.%20Data%20Cleaning.ipynb) |

**Progress** &nbsp; ▰▰▰▱▱▱▱▱  **3 / 8**

---

## 🎯 Task requirements

- [x] **Mock stream** — generate drivers with Faker: `name`, `car_number`, `experience`, `rating`, `timestamp`
- [x] Write it to **`bronze.drivers`** with **checkpointing** in the Volume, and verify records are ingesting
- [x] **Schema evolution** — add a `phone` column to the generator, restart the query, let the table accept it
- [x] **Stop** the stream, read the table with **`VERSION AS OF`** (before `phone`), then **`RESTORE`** it

## 🧩 How it is solved

```text
 Faker waves ──▶ landing/json/*.json ──▶ Auto Loader ──▶ bronze.drivers (Delta)
                                              │
                                              └── checkpoint: /Volumes/…/_checkpoints/bronze/drivers
```

| Requirement | Implementation |
|:--|:--|
| Mock stream | `DriverGenerator` drops a new JSON file ("wave") into the Volume every few seconds |
| Streaming write | `DriverStreamingIngestionService` — Auto Loader → Delta, `outputMode("append")` |
| Checkpointing | `…/landing/_checkpoints/bronze/drivers/{data,schema}` inside the Volume |
| Schema evolution | `cloudFiles.schemaEvolutionMode = addNewColumns` **+** Delta `mergeSchema = true` |
| Time travel | `SELECT … VERSION AS OF <n>` · `RESTORE TABLE … TO VERSION AS OF <n>` |

> 💡 **How "continuous" works here** — the query uses `trigger(availableNow=True)`: it drains every new file and
> then stops, resuming from the checkpoint on the next call. Calling it after each wave is the serverless-friendly
> equivalent of an always-on stream: same checkpoint, same exactly-once guarantees.

## 1️⃣ Connect to Spark

In [1]:
from pyspark.sql import functions as F

from src.core.config import get_catalog, get_environment
from src.core.infra import DatabricksConnectionService

conn = DatabricksConnectionService(app_name="yandex-go-streaming-ingestion")
conn.connect()
spark = conn.spark

CATALOG = get_catalog()
print(f"✅ Connected | Spark {spark.version} | catalog = {CATALOG} | env = {get_environment()}")

2026-10-09 17:17:27.900 | INFO     | src.core.infra.connection.databricks_connector:connect:56 - SparkSession ready | app=yandex-go-streaming-ingestion | version=4.2.0 | runtime=serverless-connect


✅ Connected | Spark 4.2.0 | catalog = yandex_go_prod | env = prod


## 3️⃣ Start the stream and watch it ingest

The loop below simulates a **live feed**: every iteration a new file lands in the Volume, the stream picks it up,
and we record how the table grows.

> ℹ️ The first iteration also ingests the 50 seed drivers created in notebook 1.

In [2]:
from src.core.etl.bronze import DriverStreamingIngestionService

TARGET = f"{CATALOG}.bronze.drivers"

driver_service = DriverStreamingIngestionService(include_phone_in_base_schema=False)

driver_service.run(spark)

print("✅ Records are being ingested continuously")

2026-10-09 17:17:27.909 | INFO     | src.core.etl.bronze.driver:run:105 - Initiating streaming ingestion pipeline for yandex_go_prod.bronze.drivers
2026-10-09 17:17:27.911 | INFO     | src.core.etl.bronze.driver:extract:54 - Starting Stream Reading [json] from: /Volumes/yandex_go_prod/raw_files/landing/json
2026-10-09 17:17:28.281 | INFO     | src.core.etl.bronze.driver:load:80 - Writing continuous stream to Delta table: yandex_go_prod.bronze.drivers
2026-10-09 17:17:41.954 | INFO     | src.core.etl.bronze.driver:load:93 - Stream batch processed successfully. Query ID: caefac09-89a0-4e4e-b216-8780ebd14d3f


✅ Records are being ingested continuously


### 📍 Checkpoint files in the Volume

The checkpoint stores the stream's progress (`offsets`, `commits`, `metadata`) — this is what makes restarts
**exactly-once**.

In [3]:
checkpoint_dir = f"{driver_service.checkpoint_path}/data"
print(f"Checkpoint location: {checkpoint_dir}")

checkpoint_files = (
    spark.read.format("binaryFile")
    .option("recursiveFileLookup", "true")
    .load(checkpoint_dir)
    .select(
        F.regexp_replace("path", r".*/bronze/drivers/data/", "").alias("file"),
        F.col("length").alias("bytes"),
        "modificationTime",
    )
    .orderBy("file")
)
display(checkpoint_files.toPandas())

assert checkpoint_files.count() > 0, "No checkpoint files found"
print("✅ Checkpoint is stored in the raw_files Volume")

Checkpoint location: /Volumes/yandex_go_prod/raw_files/landing/_checkpoints/bronze/drivers/data


,file,bytes,modificationTime
0,commits/0,29,2026-10-09 14:17:38
1,commits/1,29,2026-10-09 14:17:40
2,metadata,45,2026-10-09 14:17:32
3,offsets/0,2294,2026-10-09 14:17:35
4,offsets/1,2294,2026-10-09 14:17:35
5,sources/0/metadata,133,2026-10-09 14:17:33
6,sources/0/rocksdb/0.zip,172,2026-10-09 14:17:33
7,sources/0/rocksdb/1.zip,3989,2026-10-09 14:17:35
8,sources/0/rocksdb/2.zip,4167,2026-10-09 14:17:37
9,sources/0/rocksdb/3.zip,4257,2026-10-09 14:17:38


✅ Checkpoint is stored in the raw_files Volume


### 🧾 Delta transaction log

Every micro-batch is one atomic commit (`STREAMING UPDATE`).

In [4]:
display(
    spark.sql(f"DESCRIBE HISTORY {TARGET}").select(
        "version", "timestamp", "operation", "operationMetrics"
    ).toPandas()
)
display(spark.table(TARGET).orderBy(F.col("_ingested_at").desc()).toPandas())

,version,timestamp,operation,operationMetrics
0,2,2026-10-09 14:17:39,STREAMING UPDATE,"{'numRemovedFiles': '0', 'conflictDetectionTim..."
1,1,2026-10-09 14:17:37,STREAMING UPDATE,"{'numRemovedFiles': '0', 'numOutputRows': '62'..."
2,0,2026-10-09 14:17:31,CREATE TABLE,{}


,car_number,experience,id,name,rating,timestamp,_rescued_data,_ingested_at,_source_file
0,Y674MX,11,06b0c57e-d757-49f8-aac1-77c63150e5fa,Michael Perez,4.34,2026-10-09 13:45:29.438,None,2026-10-09 14:17:34.601,/Volumes/yandex_go_prod/raw_files/landing/json...
1,M191EB,9,e6d47d34-7e54-44d7-8d15-76338379e94e,Thomas Wilson,5.80,2026-10-09 13:45:29.438,None,2026-10-09 14:17:34.601,/Volumes/yandex_go_prod/raw_files/landing/json...
2,K409BC,17,5b730cad-9176-4f62-ba2b-e4d3bfae0535,Stephen King,4.64,2026-10-09 13:45:29.438,None,2026-10-09 14:17:34.601,/Volumes/yandex_go_prod/raw_files/landing/json...
3,P194AK,25,71fad878-b369-4102-89f3-93d78b8ae077,George Blake,4.23,2026-10-09 13:45:29.439,None,2026-10-09 14:17:34.601,/Volumes/yandex_go_prod/raw_files/landing/json...
4,H616EM,21,9e6bfff5-b7a1-4d67-bf41-58639bedfe39,Virginia Wilson,4.64,2026-10-09 13:45:29.439,None,2026-10-09 14:17:34.601,/Volumes/yandex_go_prod/raw_files/landing/json...
...,...,...,...,...,...,...,...,...,...
95,M355EP,4,3d6350f0-40a5-4f52-af38-a9c7a6c9253a,Julie Berry,4.83,2026-10-09 13:45:29.421,None,2026-10-09 14:17:34.175,/Volumes/yandex_go_prod/raw_files/landing/json...
96,P786CT,20,4b6587d8-8ef4-44d1-9299-b5fdfdff95ac,Cindy Li,4.98,2026-10-09 13:45:29.422,None,2026-10-09 14:17:34.175,/Volumes/yandex_go_prod/raw_files/landing/json...
97,C795PK,6,f452cbd8-63fa-421d-8c1f-1c4fff811cc3,Richard Dawson,4.67,2026-10-09 13:45:29.422,None,2026-10-09 14:17:34.175,/Volumes/yandex_go_prod/raw_files/landing/json...
98,E592CP,9,81c1e7ff-6efa-4d5b-9988-5afcbb61a9cd,Victoria Brown,5.80,2026-10-09 13:45:29.423,None,2026-10-09 14:17:34.175,/Volumes/yandex_go_prod/raw_files/landing/json...


## 4️⃣ Schema evolution — add the `phone` column

**Scenario:** a new upstream version starts sending a `phone` number.

1. Remember the table state **before** the change — we will travel back to it in step 6.
2. Update the generator (`with_phone=True`) and let a file with the new column land.
3. Restart the query with the `phone` column in its schema hints.

```text
 v0 … vN   id · name · car_number · experience · rating · timestamp · _ingested_at · _source_file
 vN+1 …    + phone                                  ← table schema widened by mergeSchema
```

#### 📌 Remember the table state before `phone`

`DESCRIBE HISTORY` gives the latest Delta version — the last one that has **no** `phone` column. We keep it for the time-travel step.

In [5]:
from pyspark.sql import functions as F

version_before_phone = spark.sql(f"DESCRIBE HISTORY {TARGET}").agg(F.max("version")).first()[0]
rows_before_phone = spark.table(TARGET).count()
columns_before_phone = spark.table(TARGET).columns

assert "phone" not in columns_before_phone
print(f"📌 Delta version before phone: {version_before_phone} | rows: {rows_before_phone}")
print(f"   Columns: {columns_before_phone}")

📌 Delta version before phone: 2 | rows: 100
   Columns: ['car_number', 'experience', 'id', 'name', 'rating', 'timestamp', '_rescued_data', '_ingested_at', '_source_file']


#### 🧑‍✈️ Updated Faker script — a file with the `phone` column

A new JSON file with an extra `phone` field lands in the Volume; the stream restart below picks it up.

In [6]:
from src.core.utils.data_generation.driver import DriverGenerator

driver_gen = DriverGenerator(subfolder="json", volume_name="landing", pool_size=100)
driver_gen.generate(spark, row_count=20, with_phone=True)

print(f"✅ File with `phone` written to: {driver_gen.output_path}")

✅ File with `phone` written to: /Volumes/yandex_go_prod/raw_files/landing/json


In [7]:
from src.core.etl.bronze import DriverStreamingIngestionService

driver_service = DriverStreamingIngestionService(include_phone_in_base_schema=True)

driver_service.run(spark)

print("✅ Records are being ingested continuously")

2026-10-09 17:17:53.500 | INFO     | src.core.etl.bronze.driver:run:105 - Initiating streaming ingestion pipeline for yandex_go_prod.bronze.drivers
2026-10-09 17:17:53.501 | INFO     | src.core.etl.bronze.driver:extract:54 - Starting Stream Reading [json] from: /Volumes/yandex_go_prod/raw_files/landing/json
2026-10-09 17:17:53.689 | INFO     | src.core.etl.bronze.driver:load:80 - Writing continuous stream to Delta table: yandex_go_prod.bronze.drivers
2026-10-09 17:18:04.061 | INFO     | src.core.etl.bronze.driver:load:93 - Stream batch processed successfully. Query ID: caefac09-89a0-4e4e-b216-8780ebd14d3f


✅ Records are being ingested continuously


## 5️⃣ Stop the stream

`availableNow` queries stop on their own once they are caught up. The cell below makes that explicit and
cleans up anything that might still be running.

In [8]:
for active_query in spark.streams.active:
    print(f"Stopping query {active_query.name or active_query.id} ...")
    active_query.stop()

print(f"Active streaming queries: {len(spark.streams.active)}")
assert len(spark.streams.active) == 0
print("✅ Stream stopped")

Active streaming queries: 0
✅ Stream stopped


## 6️⃣ Time Travel and `RESTORE`

Delta keeps every version of the table. We query the state **before** `phone` existed and then roll the table back.

```sql
-- read an old snapshot
SELECT * FROM <catalog>.bronze.drivers VERSION AS OF <n>;

-- permanently roll the table back
RESTORE TABLE <catalog>.bronze.drivers TO VERSION AS OF <n>;
```

### 🕰 Full history

In [9]:
display(
    spark.sql(f"DESCRIBE HISTORY {TARGET}").select(
        "version", "timestamp", "operation", "operationParameters", "operationMetrics"
    ).toPandas()
)

,version,timestamp,operation,operationParameters,operationMetrics
0,4,2026-10-09 14:18:01,STREAMING UPDATE,"{'outputMode': 'Append', 'queryId': 'caefac09-...","{'numRemovedFiles': '0', 'numOutputRows': '20'..."
1,3,2026-10-09 14:17:57,STREAMING UPDATE,"{'outputMode': 'Append', 'queryId': 'caefac09-...",{}
2,2,2026-10-09 14:17:39,STREAMING UPDATE,"{'outputMode': 'Append', 'queryId': 'caefac09-...","{'numRemovedFiles': '0', 'conflictDetectionTim..."
3,1,2026-10-09 14:17:37,STREAMING UPDATE,"{'outputMode': 'Append', 'queryId': 'caefac09-...","{'numRemovedFiles': '0', 'numOutputRows': '62'..."
4,0,2026-10-09 14:17:31,CREATE TABLE,"{'partitionBy': '[]', 'clusterBy': '[]', 'desc...",{}


### ⏪ `VERSION AS OF` — the table before `phone`

The version number comes from `DESCRIBE HISTORY` (captured before the `phone` file was ingested).

In [10]:
before_df = spark.sql(f"SELECT * FROM {TARGET} VERSION AS OF {version_before_phone}")

assert "phone" not in before_df.columns, "Old snapshot must not contain `phone`"
assert before_df.count() == rows_before_phone
print(f"✅ VERSION AS OF {version_before_phone}: {before_df.count()} rows | columns: {before_df.columns}")
display(before_df.toPandas())

✅ VERSION AS OF 2: 100 rows | columns: ['car_number', 'experience', 'id', 'name', 'rating', 'timestamp', '_rescued_data', '_ingested_at', '_source_file']


,car_number,experience,id,name,rating,timestamp,_rescued_data,_ingested_at,_source_file
0,B606TO,1,b0873a0f-0334-4cca-b4a8-13d25570084b,Julia Jensen,4.88,2026-10-09 13:46:04.036,None,2026-10-09 14:17:34.175,/Volumes/yandex_go_prod/raw_files/landing/json...
1,B884PE,11,fb50c670-9e54-4e19-a6c8-78bab694de26,Patricia Bryant,4.39,2026-10-09 13:46:04.036,None,2026-10-09 14:17:34.175,/Volumes/yandex_go_prod/raw_files/landing/json...
2,A864KO,15,b4f741aa-e8c7-4c9e-92f1-3d67d8ed6793,Mr. Tyler Pierce,4.67,2026-10-09 13:46:04.036,None,2026-10-09 14:17:34.175,/Volumes/yandex_go_prod/raw_files/landing/json...
3,E308MK,2,da3b3021-9e1d-4594-b168-ecb505d8ab8c,Barbara Newton,4.47,2026-10-09 13:46:04.036,None,2026-10-09 14:17:34.175,/Volumes/yandex_go_prod/raw_files/landing/json...
4,P685XY,5,258d5a93-958e-4aa7-8dfe-519dc24c472e,Kimberly Smith,4.54,2026-10-09 13:46:04.036,None,2026-10-09 14:17:34.175,/Volumes/yandex_go_prod/raw_files/landing/json...
...,...,...,...,...,...,...,...,...,...
95,Y767CE,11,06ae4148-efcc-46b6-89a9-c1e44df180d0,Brian Chen,4.63,2026-10-09 13:45:29.435,None,2026-10-09 14:17:34.601,/Volumes/yandex_go_prod/raw_files/landing/json...
96,K368TX,8,9b410f0b-31b7-424e-860b-daad7447a088,Susan Smith,4.64,2026-10-09 13:45:29.436,None,2026-10-09 14:17:34.601,/Volumes/yandex_go_prod/raw_files/landing/json...
97,P608AB,23,8646b8ad-4f52-4577-a2cb-b9ab3819fe4d,Barry Fuller,4.88,2026-10-09 13:45:29.436,None,2026-10-09 14:17:34.601,/Volumes/yandex_go_prod/raw_files/landing/json...
98,P653HP,20,5b730cad-9176-4f62-ba2b-e4d3bfae0535,Albert Miller,4.39,2026-10-09 13:45:29.437,None,2026-10-09 14:17:34.601,/Volumes/yandex_go_prod/raw_files/landing/json...


### ♻️ `RESTORE` — roll the table back permanently

```sql
RESTORE TABLE <catalog>.bronze.drivers TO VERSION AS OF <n>;
```

In [11]:
print(f"Columns before RESTORE: {spark.table(TARGET).columns}")

spark.sql(f"RESTORE TABLE {TARGET} TO VERSION AS OF {version_before_phone}")

print("✅ RESTORE executed")

Columns before RESTORE: ['car_number', 'experience', 'id', 'name', 'rating', 'timestamp', '_rescued_data', '_ingested_at', '_source_file', 'phone']
✅ RESTORE executed


### 🔍 Schema check after `RESTORE`

In [12]:
restored_df = spark.table(TARGET)

print(f"Columns after RESTORE: {restored_df.columns}")
print(f"Rows after RESTORE   : {restored_df.count()}")

assert restored_df.columns == columns_before_phone, "Schema must match the pre-phone version"
assert "phone" not in restored_df.columns, "`phone` must be gone after RESTORE"
assert restored_df.count() == rows_before_phone
print(f"✅ Table restored to version {version_before_phone}; schema is back to the pre-phone state")

display(
    spark.sql(f"DESCRIBE HISTORY {TARGET} LIMIT 3").select(
        "version", "timestamp", "operation", "operationParameters"
    ).toPandas()
)

Columns after RESTORE: ['car_number', 'experience', 'id', 'name', 'rating', 'timestamp', '_rescued_data', '_ingested_at', '_source_file']
Rows after RESTORE   : 100
✅ Table restored to version 2; schema is back to the pre-phone state


,version,timestamp,operation,operationParameters
0,5,2026-10-09 14:18:14,RESTORE,"{'version': '2', 'timestamp': None}"
1,4,2026-10-09 14:18:01,STREAMING UPDATE,"{'outputMode': 'Append', 'queryId': 'caefac09-..."
2,3,2026-10-09 14:17:57,STREAMING UPDATE,"{'outputMode': 'Append', 'queryId': 'caefac09-..."


#### I included an example of the RESTORE table command here because the phone column is useful downstream. This approach also allows us to avoid building a data backfill pipeline and losing phones of drivers, which are already ingested

---

## ✅ Checkpoint

- [x] Faker stream written to `bronze.drivers` with a checkpoint in the Volume
- [x] Row counts grow after every wave — ingestion verified
- [x] `phone` column added via schema evolution
- [x] `VERSION AS OF` shows the pre-`phone` snapshot
- [x] `RESTORE` rolled the table back; schema verified

> **Next up ▶ [4 · Data Cleaning](./4.%20Data%20Cleaning.ipynb)** — clean the Bronze data and build the Silver layer.

<div align="center"><sub>Yandex Go Analytics · notebook 3 of 8 · <a href="https://github.com/stanislavpanfilenko/YandexGoAnalytics">GitHub</a>